In [1]:
import numpy as np
import pandas as pd
import tkinter as tk
import time
import random

from pathlib import Path
from scipy.io import wavfile

import sounddevice as sd

In [2]:
stimulus_folder = Path("../stimuli/final_27")

wav_lookup = {}

for file in stimulus_folder.glob("*.wav"):

    stim_id = file.name.split("_")[0]

    wav_lookup[stim_id] = str(file)

print("Number of stimuli:", len(wav_lookup))

print(wav_lookup["S01"])

Number of stimuli: 27
..\stimuli\final_27\S01_T2_IFI50_AM30.wav


In [3]:
stimuli = [f"S{i:02d}" for i in range(1, 28)]

trials = []

for i in range(len(stimuli)):
    for j in range(i, len(stimuli)):

        trials.append({
            "stimulus_A": stimuli[i],
            "stimulus_B": stimuli[j]
        })

print("Total trials:", len(trials))

random.shuffle(trials)

trial_df = pd.DataFrame(trials)

trial_df.head()

Total trials: 378


,stimulus_A,stimulus_B
0,S07,S09
1,S10,S25
2,S03,S11
3,S23,S27
4,S07,S18


In [4]:
blocks = [
    trial_df.iloc[:75].reset_index(drop=True),
    trial_df.iloc[75:150].reset_index(drop=True),
    trial_df.iloc[150:225].reset_index(drop=True),
    trial_df.iloc[225:300].reset_index(drop=True),
    trial_df.iloc[300:].reset_index(drop=True)
]

for i, block in enumerate(blocks):
    print(
        f"Block {i+1}: {len(block)}"
    )

Block 1: 75
Block 2: 75
Block 3: 75
Block 4: 75
Block 5: 78


In [5]:
results_dir = Path("../results")

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(results_dir)

..\results


In [19]:
# ==========================================
# CLEAR SCREEN
# ==========================================

def clear_widgets(root):

    for widget in root.winfo_children():

        widget.destroy()


# ==========================================
# PLAY WAV
# ==========================================

def play_wav(filepath):

    fs, data = wavfile.read(filepath)

    sd.play(
        data,
        samplerate=fs,
        blocking=True
    )


# ==========================================
# RESPONSE COLLECTION
# ==========================================

def collect_response(root):

    response = []
    rt = []

    start_time = time.time()

    def on_key(event):

        if event.keysym == "End":

            if len(response) == 0:

                response.append(1)

                rt.append(
                    (time.time() - start_time) * 1000
                )

        elif event.keysym == "Next":

            if len(response) == 0:

                response.append(0)

                rt.append(
                    (time.time() - start_time) * 1000
                )
        

    root.bind("<Key>", on_key)

    while time.time() - start_time < 2.0:

        root.update()

    root.unbind("<Key>")

    if len(response) == 0:

        return None, np.nan

    return response[0], rt[0]


# ==========================================
# SINGLE TRIAL
# ==========================================

def run_trial(
    root,
    stimA,
    stimB,
    wav_lookup
):

    clear_widgets(root)

    fixation = tk.Label(
        root,
        text="+",
        font=("Arial",36),
        fg="white",
        bg="black"
    )

    fixation.pack(expand=True)

    root.update()

    time.sleep(0.3)

    fixation.destroy()

    # Stimulus A

    play_wav(
        wav_lookup[stimA]
    )

    # ISI

    time.sleep(0.150)

    # Stimulus B

    play_wav(
        wav_lookup[stimB]
    )

    # Response Window

    clear_widgets(root)

    root.focus_force()

    root.update()

    response, rt = collect_response(root)

    # ITI

    time.sleep(0.150)

    return response, rt


# ==========================================
# MATRIX UPDATE
# ==========================================

def update_matrix(
    matrix,
    stimA,
    stimB,
    response
):

    i = int(stimA[1:]) - 1
    j = int(stimB[1:]) - 1

    matrix[i,j] = response
    matrix[j,i] = response

    return matrix


# ==========================================
# BLOCK BREAK
# ==========================================

def block_break(root, block_num):

    clear_widgets(root)

    label = tk.Label(
        root,
        text=f"End of Block {block_num}\n\nTake a short break\n\nPress SPACE to continue",
        font=("Arial",24),
        fg="white",
        bg="black"
    )

    label.pack(expand=True)

    waiting = [True]

    def continue_exp(event):

        if event.keysym == "space":

            waiting[0] = False

    root.bind("<Key>", continue_exp)

    while waiting[0]:

        root.update()

    root.unbind("<Key>")


# ==========================================
# END SCREEN
# ==========================================

def end_screen(root):

    clear_widgets(root)

    label = tk.Label(
        root,
        text="Experiment Complete\n\nThank You",
        font=("Arial",28),
        fg="white",
        bg="black"
    )

    label.pack(expand=True)

    root.update()

    time.sleep(3)

In [20]:
# ==========================================
# PARTICIPANT ID
# ==========================================

participant_id = input(
    "Participant ID: "
)

# ==========================================
# TEST BLOCK
# ==========================================

test_block = blocks[0].head(3)

# ==========================================
# STORAGE
# ==========================================

trial_results = []

participant_matrix = np.full(
    (27,27),
    np.nan
)

# ==========================================
# WINDOW
# ==========================================

root = tk.Tk()

root.configure(
    bg="black"
)

root.geometry(
    "800x600"
)

root.focus_force()

# ==========================================
# RUN 3 TRIALS
# ==========================================

for trial_num, row in test_block.iterrows():

    stimA = row["stimulus_A"]
    stimB = row["stimulus_B"]

    response, rt = run_trial(
        root,
        stimA,
        stimB,
        wav_lookup
    )

    update_matrix(
        participant_matrix,
        stimA,
        stimB,
        response
    )

    trial_results.append({

        "participant":
            participant_id,

        "block":
            1,

        "trial":
            trial_num + 1,

        "stimulus_A":
            stimA,

        "stimulus_B":
            stimB,

        "response":
            response,

        "rt_ms":
            rt
    })

# ==========================================
# SAVE TRIAL DATA
# ==========================================

trial_df = pd.DataFrame(
    trial_results
)

trial_df.to_csv(
    results_dir /
    f"participant_{participant_id}_trials.csv",
    index=False
)

# ==========================================
# SAVE MATRIX
# ==========================================

matrix_df = pd.DataFrame(
    participant_matrix
)

matrix_df.to_csv(
    results_dir /
    f"participant_{participant_id}_matrix.csv",
    index=False
)

# ==========================================
# END
# ==========================================

end_screen(root)

root.destroy()

print(
    "3 Trial Test Complete"
)

3 Trial Test Complete


In [ ]:
# ==========================================
# PARTICIPANT ID
# ==========================================

participant_id = input(
    "Participant ID: "
)

# ==========================================
# STORAGE
# ==========================================

trial_results = []

participant_matrix = np.full(
    (27,27),
    np.nan
)

# ==========================================
# WINDOW
# ==========================================

root = tk.Tk()

root.configure(
    bg="black"
)

root.geometry(
    "800x600"
)

root.focus_force()

# ==========================================
# CHECK
# ==========================================

print(
    "Total Trials:",
    sum(len(b) for b in blocks)
)

# ==========================================
# FULL EXPERIMENT
# ==========================================

for block_num, block in enumerate(
    blocks,
    start=1
):

    print(
        f"Starting Block {block_num}"
    )

    for trial_num, row in block.iterrows():

        stimA = row["stimulus_A"]
        stimB = row["stimulus_B"]

        response, rt = run_trial(
            root,
            stimA,
            stimB,
            wav_lookup
        )

        update_matrix(
            participant_matrix,
            stimA,
            stimB,
            response
        )

        trial_results.append({

            "participant":
                participant_id,

            "block":
                block_num,

            "trial":
                trial_num + 1,

            "stimulus_A":
                stimA,

            "stimulus_B":
                stimB,

            "response":
                response,

            "rt_ms":
                rt
        })

    # --------------------------
    # BLOCK BREAK
    # --------------------------

    if block_num < 5:

        block_break(
            root,
            block_num
        )

# ==========================================
# SAVE TRIAL DATA
# ==========================================

trial_df = pd.DataFrame(
    trial_results
)

trial_df.to_csv(
    results_dir /
    f"participant_{participant_id}_trials.csv",
    index=False
)

# ==========================================
# SAVE MATRIX
# ==========================================

matrix_df = pd.DataFrame(
    participant_matrix
)

matrix_df.to_csv(
    results_dir /
    f"participant_{participant_id}_matrix.csv",
    index=False
)

# ==========================================
# END SCREEN
# ==========================================

end_screen(root)

root.destroy()

print(
    "Experiment Complete"
)